# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 5 — Calibrated Day-Ahead Intervals

**Duration** 50 minutes · **Work in pairs** · `git checkout lab5-solution`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Find the **two bugs** in the starter interval you inherited. Note both in `INTERVALS.md`. |
| 10 | Reuse the Lab 4 feature matrix. Split train / calibration / test **by time**. Fit `QuantileLGBM` at τ ∈ {0.05, 0.50, 0.95} and repair quantile crossing. |
| 10 | Coverage and mean width of the raw band on the test window. It will come out **below** 90%. |
| 10 | Calibrate with `CQR` on the held-out October window. Re-apply. Coverage should snap to ≈ 0.90. |
| 10 | Report coverage **marginally and conditionally** with `coverage_report`. Pinball losses and the Winkler score. Plot one calibrated day. |
| 5 | Recommend one method in `INTERVALS.md`. Commit. |

> **Day boundary.** Tasks 1–3 close Day 2; the room goes home holding an interval that
> visibly under-covers. Tasks 4–6 open Day 3.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar
from tayyar.features.build import build_features

df = join_calendar(load_demand(f"{DATA}/ksa_grid_demand.csv"),
                   load_calendar(f"{DATA}/ksa_calendar.csv"))
feat = build_features(df[["demand_mw", "temp_c", "is_weekend",
                          "is_ramadan", "is_eid", "is_national_day"]])
Xcols = [c for c in feat.columns if c != "demand_mw"]
X, yy = feat[Xcols].astype(float), feat["demand_mw"]

CUT_TRAIN, CUT_CAL = "2023-09-30 23:00", "2023-10-31 23:00"
print(f"{len(feat):,} rows x {len(Xcols)} features (the Lab 4 design matrix)")

### Task 1 — the starter interval you inherited

Run the next cell. It produces a 90% band and a coverage number. Five minutes: name the two
bugs precisely, and write them into `INTERVALS.md`. Do not fix them yet.

In [ ]:
# ---------------------------------------------------------------------------
# The starter code you inherited
#
# "Day-ahead point forecast with a 90% interval. I checked the coverage."
#
# It runs and it produces an interval. Two lines make the reported coverage
# worthless — one about WHAT is being estimated, one about WHERE it is measured.
# ---------------------------------------------------------------------------
import lightgbm as lgb

Y24 = yy.shift(-24)                                  # the day-ahead target
ok = X.notna().all(axis=1) & Y24.notna()
Xfit, Yfit = X[ok][:CUT_TRAIN], Y24[ok][:CUT_TRAIN]

point = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=63,
                          verbose=-1, random_state=212).fit(Xfit, Yfit)

fitted = point.predict(Xfit)
resid = Yfit - fitted
se = resid.std() / np.sqrt(len(resid))               # "the standard error"
half = 1.645 * se                                    # "90%"

lo_s, hi_s = fitted - half, fitted + half
cov_s = float(np.mean((Yfit >= lo_s) & (Yfit <= hi_s)))

print(f"residual sd            : {resid.std():8,.1f} MW")
print(f"reported 90% band width: {2 * half:8,.1f} MW")
print(f"reported coverage      : {cov_s:8.3f}")

**Bug (a) — it is a confidence interval, not a prediction interval.**
`resid.std() / np.sqrt(n)` is the standard error of the *mean*. It answers "where is the
conditional average?" and it shrinks towards zero as n grows — with 23,700 training rows the
band comes out about 13 MW wide on a 30,000 MW system. The control room is not asking about the
average of a hypothetical population of tomorrows; it is asking where *tomorrow* will land, so
the interval must also carry the irreducible noise of a single observation. That is a
prediction interval, and it does not shrink with n.

**Bug (b) — coverage is measured on the training set.** `fitted` are in-sample predictions from
a 300-tree model, so the residuals it is scored against are the residuals it was fitted to
minimise. Whatever coverage that reports is an upper bound on the truth, not an estimate of it.

Bug (a) is loud: a 13 MW band is visibly absurd the moment you read the width. Bug (b) is
silent — nothing in that printout tells you the coverage figure was measured on the very rows
the model was fitted to minimise error on, and a healthy-looking 0.90 from a training set is
how this defect actually ships. Separate the two and both become measurable.

In [ ]:
Xtest, Ytest = X[ok][CUT_CAL:], Y24[ok][CUT_CAL:]
pred_test = point.predict(Xtest)

def cov(y, lo, hi):
    return float(np.mean((np.asarray(y) >= lo) & (np.asarray(y) <= hi)))

rows = []
for label, hw in [("(a) confidence: 1.645 * sd / sqrt(n)", 1.645 * se),
                  ("(b) prediction: 1.645 * sd",           1.645 * resid.std())]:
    rows.append({"band": label, "width_MW": round(2 * hw, 1),
                 "coverage_TRAIN": round(cov(Yfit, fitted - hw, fitted + hw), 3),
                 "coverage_TEST":  round(cov(Ytest, pred_test - hw, pred_test + hw), 3)})
print(pd.DataFrame(rows).to_string(index=False))

Read the table by column. Down the first column, bug (a): the confidence band is
narrower than the prediction band by a factor of √n and covers almost nothing. Across the
row, bug (b): the honest prediction band covers close to nominal on the rows it was fitted on
and materially less on rows it has never seen. That gap is the entire reason the calibration
window in Task 2 is held out.

A third defect follows from bug (a)'s formula and is worth naming: a constant half-width
assumes the uncertainty is the same at 04:00 in February and at 15:00 in July. It is not.
That is what the quantile model in Task 2 fixes.

**`INTERVALS.md`, section 1**

```
## Bugs in the inherited starter (labs/lab5)

(a) Confidence interval where a prediction interval is required.
    half = 1.645 * resid.std() / sqrt(n) is the standard error of the MEAN.
    It shrinks like 1/sqrt(n) and omits the irreducible noise of a single
    future observation. Correct object: a prediction interval, which does not
    shrink to zero.

(b) Coverage estimated on the training set.
    `fitted` are in-sample predictions of a 300-tree model, scored against the
    residuals it was fitted to minimise. The figure is an upper bound, not an
    estimate. Correct: a held-out calibration window, disjoint in time.
```

### Task 2 — three quantiles, split by time, de-crossed

Three time blocks, disjoint and in order:

| block | window | purpose |
|---|---|---|
| train | … through 2023-09-30 | fit the quantile models |
| calibration | October 2023 | estimate q̂ — never used for fitting |
| test | November–December 2023 | the only honest coverage estimate |

`QuantileLGBM` fits one model per (horizon, τ): 3 × 24 = 72 models. Nothing constrains the
three fits to be monotone, so q95 can land below q50 for some rows — an incoherent
distribution that would produce a negative-width interval. Sort each row.

In [ ]:
from tayyar.models.quantile import QuantileLGBM

Xtr = X[:CUT_TRAIN].dropna(); ytr = yy[Xtr.index]
Xca, yca = X[CUT_TRAIN:CUT_CAL], yy[CUT_TRAIN:CUT_CAL]
Xte, yte = X[CUT_CAL:], yy[CUT_CAL:]
for nm, ix in (("train", Xtr.index), ("calib", Xca.index), ("test", Xte.index)):
    print(f"{nm:<6} {ix.min():%Y-%m-%d} -> {ix.max():%Y-%m-%d}  ({len(ix):,} rows)")

TAUS = (0.05, 0.50, 0.95)
qm = QuantileLGBM(quantiles=TAUS, horizon=24).fit(Xtr, ytr)   # 72 models — a few minutes
print(f"\nfitted {len(qm.models_)} models = {len(TAUS)} quantiles x 24 horizons")

In [ ]:
def q_predict(Xs):
    """(3, n_origins, 24) — one plane per quantile, one row per origin."""
    return np.stack([np.column_stack([qm.models_[(h, t)].predict(Xs)
                                      for h in range(1, 25)]) for t in TAUS])

def day_blocks(Xs, ys, step=24):
    """One forecast origin per day; the 24 actual hours that follow each."""
    rows = list(range(0, len(Xs) - 24, step))
    Y = np.array([ys.iloc[i + 1: i + 25].to_numpy() for i in rows])
    return Y, rows, [Xs.index[i] for i in rows]

Yca, rows_ca, idx_ca = day_blocks(Xca, yca)
Pca = q_predict(Xca.iloc[rows_ca])
Yte, rows_te, idx_te = day_blocks(Xte, yte)
Pte = q_predict(Xte.iloc[rows_te])

for nm, P in (("calibration", Pca), ("test", Pte)):
    print(f"{nm:<12} {P.shape[1]} origins x 24 h = {P[0].size} cells")
    print(f"   q05 > q50 : {int((P[0] > P[1]).sum()):4d}")
    print(f"   q50 > q95 : {int((P[1] > P[2]).sum()):4d}")
    print(f"   q05 > q95 : {int((P[0] > P[2]).sum()):4d}   <- band inversions")

# The repair. Sorting the row is the blunt version and is what
# QuantileLGBM.predict_interval does. Here the band itself is never inverted
# (q05 <= q95 in every cell), so the only incoherence is the median falling
# outside its own interval — clip it back in and keep the model's endpoints.
Pca_raw, Pte_raw = Pca.copy(), Pte.copy()
for P in (Pca, Pte):
    assert (P[0] <= P[2]).all()
    P[1] = np.clip(P[1], P[0], P[2])
print(f"\nmedians clipped back inside the band, largest move: "
      f"{np.abs(Pte - Pte_raw).max():,.1f} MW")
print("remaining incoherent cells:",
      int((Pte[0] > Pte[1]).sum() + (Pte[1] > Pte[2]).sum()))

Eighty incoherent cells out of 1,464 on the test window: 17 where q05 came out above
q50, 63 where q50 came out above q95. Nothing constrains three separate fits to be monotone,
so this is expected rather than surprising — and "rare and small" is still not "acceptable".
A median sitting outside its own 90% interval is not a wide forecast or a noisy one; it is
not a distribution, and it is indefensible on a control-room screen.

Two repairs. `np.sort(P, axis=0)` takes the order statistics and is what
`QuantileLGBM.predict_interval` does — correct, and the right choice when the band endpoints
themselves invert. Here they never do (`q05 > q95` in zero cells), so the band the model
learned is coherent and only the median is misplaced; clipping the median back inside keeps
the endpoints the quantile models actually produced. The principled fix, ahead of both, is a
monotone constraint at fit time.

### Task 3 — what the raw band actually covers

In [ ]:
from tayyar.eval.pinball import coverage

lo_raw, med, hi_raw = Pte[0], Pte[1], Pte[2]
print(f"nominal coverage        : 0.900")
print(f"empirical coverage      : {coverage(Yte.ravel(), lo_raw.ravel(), hi_raw.ravel()):.4f}")
print(f"mean width              : {np.mean(hi_raw - lo_raw):,.1f} MW")
print()
print(f"on the calibration month: "
      f"{coverage(Yca.ravel(), Pca[0].ravel(), Pca[2].ravel()):.4f}")

**0.786 against a nominal 0.900.** One test hour in five falls outside a band advertised as
covering nineteen in twenty.

Nothing is broken. `QuantileLGBM` minimises pinball loss on the *training* window, and a
finite, regularised, boosted model does not reproduce its training quantiles out of sample —
gradient boosting shrinks towards the centre, the test window is a different season from most
of the training data, and the pinball optimum is only asymptotically the quantile. The
calibration month shows the same shortfall (0.823), which is what makes it usable as a
correction: the gap is a property of the model, not of November.

**This is where Day 2 ends.** Go home holding a 90% interval that covers 79%.

### Task 4 — conformal calibration (CQR)

The recipe, in four lines:

1. **Split.** Hold out a calibration window the model never saw — October here.
2. **Score.** For CQR the nonconformity score is `max(lo - y, y - hi)`: how far the truth fell
   *outside* the band. It is negative when the truth was comfortably inside, which is how the
   band can also be made *narrower* when the model is over-cautious.
3. **q̂.** Take the `ceil((n+1)(1-α))`-th smallest score. The `+1` is the finite-sample
   correction — it is what turns an asymptotic statement into a guarantee that holds at any n.
4. **Widen.** The calibrated interval is `[lo - q̂, hi + q̂]`.

CQR keeps the *shape* the quantile model learned — wide on volatile afternoons, narrow at
04:00 — and rescales it. A symmetric split-conformal band around the point forecast would hit
the same marginal coverage with a constant width, and be badly wrong in both directions at
different hours.

In [ ]:
from tayyar.models.conformal import CQR, conformal_quantile

cqr = CQR(alpha=0.10).calibrate(Yca.ravel(), Pca[0].ravel(), Pca[2].ravel())
lo_cal, hi_cal = cqr.interval(lo_raw, hi_raw)

n = Yca.size
scores = np.maximum(Pca[0].ravel() - Yca.ravel(), Yca.ravel() - Pca[2].ravel())
k = int(np.ceil((n + 1) * 0.90))
print(f"calibration scores      : n = {n}")
print(f"rank taken, ceil((n+1)(1-a)) = {k}   ({k}/{n} = {k/n:.4f})")
print(f"q-hat                   : {cqr.qhat_:,.1f} MW")
print(f"  (identical to conformal_quantile: {conformal_quantile(scores, 0.10):,.1f})")
print()
print(f"coverage before         : {coverage(Yte.ravel(), lo_raw.ravel(), hi_raw.ravel()):.4f}"
      f"   mean width {np.mean(hi_raw - lo_raw):8,.1f} MW")
print(f"coverage after          : {coverage(Yte.ravel(), lo_cal.ravel(), hi_cal.ravel()):.4f}"
      f"   mean width {np.mean(hi_cal - lo_cal):8,.1f} MW")

0.786 -> **0.902**, for 409 MW added to each side and a mean width of 3,235 MW instead of
2,418. That is the trade, stated honestly: coverage was bought with sharpness, and the price
is on the table rather than hidden in a footnote.

Note what was *not* required. No distributional assumption, no normality, no correctly
specified variance model — only that the calibration and test scores be exchangeable. The
guarantee is finite-sample and distribution-free.

### Task 5 — marginal is not conditional

In [ ]:
from tayyar.models.conformal import coverage_report
from tayyar.eval.pinball import pinball_loss, interval_score

h = np.tile(np.arange(1, 25), len(Yte))          # horizon 1..24 from a 23:00 origin
band = pd.Series(np.where(pd.Series(h).isin(range(12, 19)),
                          "12:00-18:00 (peak)", "other hours"))

print("AFTER calibration")
print(coverage_report(Yte.ravel(), lo_cal.ravel(), hi_cal.ravel(), band).to_string(index=False))
print("\nBEFORE calibration")
print(coverage_report(Yte.ravel(), lo_raw.ravel(), hi_raw.ravel(), band).to_string(index=False))

In [ ]:
print(f"pinball q05 (calibrated lo) : {pinball_loss(Yte.ravel(), lo_cal.ravel(), 0.05):7.1f}")
print(f"pinball q50 (as fitted)     : {pinball_loss(Yte.ravel(), Pte_raw[1].ravel(), 0.50):7.1f}")
print(f"pinball q50 (de-crossed)    : {pinball_loss(Yte.ravel(), med.ravel(), 0.50):7.1f}")
print(f"pinball q95 (calibrated hi) : {pinball_loss(Yte.ravel(), hi_cal.ravel(), 0.95):7.1f}")
print(f"Winkler                     : "
      f"{interval_score(Yte.ravel(), lo_cal.ravel(), hi_cal.ravel(), 0.10):7.1f}")

In [ ]:
k = 3
tt = pd.date_range(idx_te[k] + pd.Timedelta(1, "h"), periods=24, freq="h")

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.fill_between(tt, lo_cal[k], hi_cal[k], color="#8FAADC", alpha=.55,
                label="90% interval (CQR-calibrated)")
ax.plot(tt, lo_raw[k], color="#7F7F7F", lw=.9, ls=":")
ax.plot(tt, hi_raw[k], color="#7F7F7F", lw=.9, ls=":", label="raw quantile band")
ax.plot(tt, med[k], color="#C00000", lw=1.8, label="forecast (q50)")
ax.plot(tt, Yte[k], color="#1F3864", lw=1.8, ls="--", label="actual")
ax.set_title(f"Day-ahead forecast from {idx_te[k]:%Y-%m-%d %H:%M} "
             f"with a calibrated 90% prediction interval")
ax.set_ylabel("MW"); ax.legend(frameon=False, ncols=4, fontsize=8, loc="upper left")
plt.tight_layout()

**The headline, and it must be said in exactly these words: marginal coverage of 0.90 hides
a conditional hole — only 0.845 on the summer and peak afternoons that reserve is actually
sized for.**

```
AFTER   ALL                 0.9016   width 3234.8
        12:00-18:00 (peak)  0.8454   width 3863.4
        other hours         0.9248   width 2975.9
```

The band is already 30% wider on the peak block, so the model *knows* those hours are harder —
it is simply not wide enough. Averaging across 24 hours lets the easy overnight hours
(0.925) pay for the hard afternoon ones, and the average clears 0.90. Every hour the average
buys is an hour that does not need reserve.

Before calibration the same shape is there and worse: 0.738 on peak against 0.806 elsewhere.
Conformal calibration lifted the whole curve; it did not close the gap, and it was never
going to. **Split conformal guarantees marginal coverage under exchangeability, not
conditional coverage.** If you need the guarantee to hold *per hour band*, you need a method
that adapts: **adaptive conformal inference (ACI)**, which updates q̂ online from the realised
miss rate, or **EnbPI**, which bootstraps ensemble residuals and re-estimates as the window
rolls. Both are the honest next step and both belong in the recommendation.

**The exchangeability warning.** Calibrate on peak summer and test in winter and the
assumption fails in the other direction: the summer scores are far larger than the winter
ones, q̂ is far too big, and the interval over-covers to roughly 0.99 while being much too
wide to size anything with. An interval that always contains the truth carries no
information. The calibration window must be **adjacent in regime**, not merely held out —
which is why October calibrates November, and why in production q̂ is re-estimated on a
rolling recent window rather than fixed once.

The pinball losses (95.2 / 368.4 / 120.9, and 366.3 for the de-crossed median) and the
Winkler score (4,323) are how you compare
two candidate intervals without arguing. Pinball scores each quantile as a quantile; Winkler
adds width and a miss penalty into the one number that trades coverage against sharpness.
Two methods that both cover 90% are separated by Winkler, and the sharper one ships.

### Mini-quiz — answers

1. **Prediction interval.** A confidence interval covers the conditional *mean* and shrinks
   like 1/√n towards zero; a prediction interval must also contain the irreducible noise of a
   single future observation, so it is wider and does not shrink to nothing however much data
   you have. Bug (a) in the starter was exactly this substitution.
2. **The 90th percentile.** Pinball loss is asymmetric on purpose: at τ = 0.9 an
   under-prediction is penalised nine times more than an over-prediction, and the minimiser
   is the conditional 0.9 quantile.
3. **Marginal (1−α) coverage**, under **exchangeability** of the calibration and test
   nonconformity scores. Marginal — averaged over everything. It says nothing about coverage
   on any particular subgroup, and exchangeability is a real assumption that a regime change
   breaks.
4. **They can cross.** Three separate fits share no monotonicity constraint, so q95 can land
   below q50 for some rows — an incoherent distribution. Fix by sorting each row, or by fitting
   with a monotone constraint. Never by ignoring it.
5. **The sharper one.** Coverage is the constraint; width is the objective. The Winkler
   (interval) score is the single number that trades the two, which is why we report it.

### Task 6 — `INTERVALS.md` and commit

Write the recommendation, not the method tour. Four short sections:

1. **The two bugs in the starter**, with the mechanism of each in one sentence.
2. **The numbers**: raw quantile band 0.786 coverage at 2,418 MW mean width; after CQR
   0.902 at 3,235 MW; q̂ = 409 MW.
3. **The recommendation**: quantile LightGBM calibrated by CQR on a rolling recent window,
   at α = 0.10. Reason: it keeps the shape of the quantile model (wide where the model is
   genuinely uncertain) and rescales it to hit nominal coverage, with a distribution-free
   finite-sample guarantee, for the cost of one held-out month.
4. **The caveat, in the same paragraph as the recommendation, not a footnote**: marginal
   0.902 hides 0.845 on the 12:00–18:00 peak. Reserve is sized on those hours. Ship the
   conditional table with the headline or ship neither, and move to adaptive conformal
   before this interval is used to size anything.

```
git add -A && git commit -m "feat(models): quantile LightGBM + CQR-calibrated day-ahead intervals"
```

### Instructor notes

**The day boundary is deliberate.** Stop after Task 3, with the room holding a 90% interval
that covers 78.6%. Do not let a fast pair run ahead into CQR — send them to the extension
below instead. The overnight gap is what makes the calibration land on Day 3 morning; if you
teach the fix in the same breath as the failure, it registers as a formula rather than as a
relief.

- **Branch `sim-hetero`** inflates the summer variance. The starter's analytic interval then
  covers about 90% overall and about 78% in summer — the same lesson as the peak-hour hole,
  arriving from the variance side rather than the hour-of-day side. Use it if the room accepts
  the conditional argument too readily.
- **Branch `sim-train-coverage`** reports 0.94 and delivers 0.83. It is bug (b) alone, in a
  form that looks entirely healthy on the page. Good as a five-minute opener on Day 3.
- **The coverage courtroom** (10 minutes, after Task 5). One pair defends the claim "our
  day-ahead interval has 90% coverage" — they are not lying, the marginal number is 0.902.
  Another pair cross-examines using only the held-out conditional table. The defence usually
  concedes within three questions; when it does, ask the room what the *honest* sentence would
  have been. Write the winning formulation on the board and require it in `INTERVALS.md`.
- **The elicitation drill** (5 minutes). Three decisions on the board — reserve margin, SKU
  stocking, staff roster. Each pair names the quantile that decision consumes and why: reserve
  is a high quantile because the cost of under-supply is asymmetric and severe; stocking
  balances holding cost against stock-out, so the quantile is the critical ratio and is rarely
  0.5; a roster is nearer the median because both over- and under-staffing cost roughly
  linearly. The point: **nobody consumes a point forecast.** They all consume a quantile, and
  if you do not ask which one, you have chosen 0.5 on their behalf.

**Troubleshooting**

| symptom | cause |
|---|---|
| coverage lands at 0.99 | the calibration window is from a different regime. Exchangeability failed; use an adjacent month. |
| coverage after CQR still ≈ 0.79 | q̂ applied to the wrong band, or the calibration scores were computed against sorted quantiles and the band against unsorted ones. |
| negative interval widths | quantile crossing was not repaired. Sort each row. |
| the fit takes forever | 72 models. Say so before the room starts it, and use the wait to teach the conformal recipe. |
| coverage reported on train looks perfect | that is bug (b) returning. Only the test window counts. |

**Fast finishers.** Re-run the calibration on a *summer* month (July) instead of October and
measure the over-coverage on the November–December test window — this is the exchangeability
warning as a number rather than a caution. Or implement one step of adaptive conformal: after
each day, nudge α by `γ(α_target − 1{miss})`, re-derive q̂, and plot the realised coverage over
the 61 test days converging on 0.90.